# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**K-Means clustering.** Chosen over the `impression_tier` baseline because it considers many dimensions jointly (traffic, trend, engagement, freshness) rather than one. K-Means specifically (over e.g. hierarchical clustering) was picked for interpretability — cluster centers are easy to read as "typical" pages, which matters since the output has to make sense to a human reviewer, not just perform well statistically.

In [1]:
# Path assumes this notebook lives in work/notebooks/ — adjust if you moved it.
DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv(DATA_PATH)
df.loc[df["trend_direction"].isin(["flat", "new"]), "trend_pct"] = 0.0
has_real_position = (df["avg_position"] > 0).astype(int)
df.loc[df["avg_position"] == 0, "avg_position"] = np.nan

numeric_features = [
    "word_count", "char_count", "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "content_age_days", "days_since_last_update", "ctr",
    "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct", "trend_pct",
]
X_num = df[numeric_features].copy()
for col in ["impressions_90d","clicks_90d","pageviews_90d","sessions_90d",
            "users_90d","engaged_sessions_90d","ai_sessions_90d","scroll_events_90d"]:
    X_num[col] = np.log1p(X_num[col])
X_num["word_count_was_missing"] = df["word_count"].isnull().astype(int)
X_num["char_count_was_missing"] = df["char_count"].isnull().astype(int)
X_num = X_num.fillna(X_num.median(numeric_only=True))
X_num["has_real_position"] = has_real_position
X_cat = pd.get_dummies(df[["trend_direction"]].fillna("unknown"), drop_first=True)
X = pd.concat([X_num, X_cat], axis=1).fillna(0)
Xs = StandardScaler().fit_transform(X)
print("Feature matrix ready:", Xs.shape)

Feature matrix ready: (30000, 25)


## 2. Split design

Clustering has no train/test split in the supervised sense. Instead, the honest check here is **stability**: does re-running K-Means with a different random seed produce roughly the same clusters (just possibly relabeled), or does the structure fall apart? I check this below and again more rigorously in w06_validation_audit.

## 3. Train + compare vs my baseline

In [2]:
# Path assumes this notebook lives in work/notebooks/ — adjust if you moved it.
DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv(DATA_PATH)
df.loc[df["trend_direction"].isin(["flat", "new"]), "trend_pct"] = 0.0
has_real_position = (df["avg_position"] > 0).astype(int)
df.loc[df["avg_position"] == 0, "avg_position"] = np.nan

numeric_features = [
    "word_count", "char_count", "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "content_age_days", "days_since_last_update", "ctr",
    "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct", "trend_pct",
]
X_num = df[numeric_features].copy()
for col in ["impressions_90d","clicks_90d","pageviews_90d","sessions_90d",
            "users_90d","engaged_sessions_90d","ai_sessions_90d","scroll_events_90d"]:
    X_num[col] = np.log1p(X_num[col])
X_num["word_count_was_missing"] = df["word_count"].isnull().astype(int)
X_num["char_count_was_missing"] = df["char_count"].isnull().astype(int)
X_num = X_num.fillna(X_num.median(numeric_only=True))
X_num["has_real_position"] = has_real_position
X_cat = pd.get_dummies(df[["trend_direction"]].fillna("unknown"), drop_first=True)
X = pd.concat([X_num, X_cat], axis=1).fillna(0)
Xs = StandardScaler().fit_transform(X)
print("Feature matrix ready:", Xs.shape)

k_range = range(2, 9)
inertias, sil_scores = [], []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(Xs)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(Xs, labels))
    print(f"k={k}: inertia={km.inertia_:,.0f}, silhouette={sil_scores[-1]:.3f}")

# NOTE: k=2 scored highest silhouette but just re-split by content_type-linked
# missing categorical fields when those were included as features — a false win.
# After excluding those fields (see w03_feature_leakage_check), k=7 is the best
# score that reflects genuine multi-metric behavior, not a single hidden category.
best_k = 7
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(Xs)

profile_cols = numeric_features
profile = df.groupby("cluster")[profile_cols].mean().round(2)
profile["n_pages"] = df.groupby("cluster").size()
print(profile)

print("\nBaseline (impression_tier) group sizes for comparison:")
print(df["impression_tier"].value_counts())

Feature matrix ready: (30000, 25)


k=2: inertia=621,533, silhouette=0.196


k=3: inertia=554,722, silhouette=0.182


k=4: inertia=508,828, silhouette=0.201


k=5: inertia=477,474, silhouette=0.214


k=6: inertia=451,648, silhouette=0.182


k=7: inertia=429,005, silhouette=0.218


k=8: inertia=407,822, silhouette=0.178


         word_count  char_count  impressions_90d  clicks_90d  pageviews_90d  \
cluster                                                                       
0               NaN         NaN          1909.53        2.76          13.52   
1           2845.86    18503.41         17131.06       59.74         110.46   
2           2612.25    17452.46            20.77        0.06           3.58   
3           5339.26    34778.18         29498.04      116.50         453.04   
4           2684.78    17819.09          1057.60        1.52           6.58   
5           5856.58    39782.98          6000.59       10.35         134.24   
6           2363.99    16169.17            68.32        0.19           7.65   

         sessions_90d  users_90d  engaged_sessions_90d  ai_sessions_90d  \
cluster                                                                   
0               12.69      12.39                  0.21             0.03   
1               88.88      84.92                  3.12         

**Comparison:** the baseline sorts everything into 4 traffic-only buckets. The 7-cluster solution instead separates pages that share traffic level but differ sharply in trend and engagement — e.g. two clusters both sit in high-traffic territory, but one is declining -10% with strong engagement (protect) while another is declining -28% with the weakest engagement and staleness in the whole dataset (urgent refresh candidate). The baseline would have treated both the same.

## 4. Errors and interpretation

**Where the model is weakest:** one cluster (the one with ~100% missing `word_count`) is largely defined by a data-tracking gap rather than a clean behavioral signal — it needs investigation (is this one content batch, one provider, one time period?) before it's trusted as a real archetype rather than a data-quality artifact.

**What the model leans on most:** log-transformed traffic volume and `trend_pct` dominate cluster separation — which makes sense, since those are the most variable, highest-signal columns after removing the keyword-only and redundant tier fields.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words (observed / directional / decision-support), never causal or 'predicting Google'